# 01 · The data

Generate/load the DraftZone v2 dataset and look at the national series, the spend,
and the **confound** (spend rises with seasonal demand). No truth leak — we read
`data/config.json`, never the sealed key.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from draftzone_mmm import datagen  # run `python -m draftzone_mmm.datagen` first
df = pd.read_csv('data/national_weekly.csv', parse_dates=['week'])
cfg = json.load(open('data/config.json'))
print('weeks:', len(df), '| realized confound corr(spend, season):', round(cfg['realized_confound'], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.week, df.conversions, lw=1)
ax.set_title('Weekly conversions'); ax.set_ylabel('conversions/wk'); plt.tight_layout()

In [ ]:
# total spend vs (proxy) season: the ~0.6 confound that makes attribution hard
chans = cfg['channels']
tot = sum(df[f'{c}_spend'] for c in chans)
ax = df.set_index('week')[[f'{c}_spend' for c in chans]].plot(figsize=(12,4), lw=1)
ax.set_title('Per-channel spend'); plt.tight_layout()

The causal chain is `spend -> impressions -> adstock(theta) -> Hill(half_sat, slope)
-> x beta -> contribution`, summed with baseline + controls. ~43% of conversions are
organic baseline. Each channel has a **distinct** theta/half_sat/slope so recovery is a
genuine test.